# Predicting Premier League Home Wins
## 1. Setup

In [19]:
from pathlib import Path
import urllib.request
import pandas as pd

DATA_DIR = Path("data") #stores in data
SEASONS = ["1516", "1617", "1718", "1819", "1920",
           "2021", "2122", "2223", "2324", "2425"] #which seasons ill be working with

## 2. Data loading

In [20]:
DATA_DIR.mkdir(exist_ok=True)
#download data and store as csv file
for season in SEASONS:
    file_path = DATA_DIR / f"E0_{season}.csv"
    if file_path.exists():
        print("Already have", file_path)
    else:
        url = f"https://football-data.co.uk/mmz4281/{season}/E0.csv"
        urllib.request.urlretrieve(url, file_path)
        print("Downloaded", file_path)

Already have data\E0_1516.csv
Already have data\E0_1617.csv
Already have data\E0_1718.csv
Already have data\E0_1819.csv
Already have data\E0_1920.csv
Already have data\E0_2021.csv
Already have data\E0_2122.csv
Already have data\E0_2223.csv
Already have data\E0_2324.csv
Already have data\E0_2425.csv


In [21]:
frames = []
for season in SEASONS:
    df_season = pd.read_csv(DATA_DIR / f"E0_{season}.csv") #read csv files
    df_season["Season"] = season
    frames.append(df_season)

matches = pd.concat(frames, ignore_index=True) #combine
matches.shape 

C:\Users\adam1_4sss501\AppData\Local\Temp\ipykernel_19004\3318796406.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_season["Season"] = season


(3800, 154)

## 3. Data audit

In [22]:
missing_pct = matches.isna().mean() * 100 #calculate missing values
missing_pct.sort_values(ascending=False).head(20)

1XBD       90.236842
1XBH       90.236842
1XBA       90.236842
BFE>2.5    90.078947
BFE<2.5    90.078947
BFA        90.026316
BFH        90.026316
BFD        90.026316
1XBCD      90.000000
1XBCH      90.000000
BFECH      90.000000
1XBCA      90.000000
BFEAHA     90.000000
BFCH       90.000000
BFCD       90.000000
BFCA       90.000000
BFECAHA    90.000000
BFECAHH    90.000000
BFEH       90.000000
BFEAHH     90.000000
dtype: float64

In [23]:
key_cols = ["Date", "HomeTeam", "AwayTeam", "FTR",
            "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC",
            "B365H", "B365D", "B365A",
            "AvgH", "AvgD", "AvgA", "BbAvH", "BbAvD", "BbAvA"]
missing_pct[key_cols] #show missing data percent for my columns

Date         0.0
HomeTeam     0.0
AwayTeam     0.0
FTR          0.0
FTHG         0.0
FTAG         0.0
HS           0.0
AS           0.0
HST          0.0
AST          0.0
HC           0.0
AC           0.0
B365H        0.0
B365D        0.0
B365A        0.0
AvgH        40.0
AvgD        40.0
AvgA        40.0
BbAvH       60.0
BbAvD       60.0
BbAvA       60.0
dtype: float64

The average odds column names changed between seasons (BbAvH to AvgH) so Bet 365 odds (B365H/D/A) were used instead as they are complete for all 10 seasons

In [24]:
keep_cols = ["Season", "Date", "HomeTeam", "AwayTeam", "FTR",
             "FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC",
             "B365H", "B365D", "B365A"] #removed useless columns
df = matches[keep_cols].copy()
df.head() #first five rows

,Season,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A
0,1516,08/08/2015,Bournemouth,Aston Villa,A,0,1,11,7,2,3,6,3,2.00,3.6,4.00
1,1516,08/08/2015,Chelsea,Swansea,D,2,2,11,18,3,10,4,8,1.36,5.0,11.00
2,1516,08/08/2015,Everton,Watford,D,2,2,10,11,5,5,8,2,1.70,3.9,5.50
3,1516,08/08/2015,Leicester,Sunderland,H,4,2,19,10,8,5,6,3,1.95,3.5,4.33
4,1516,08/08/2015,Man United,Tottenham,H,1,0,9,9,1,4,1,2,1.65,4.0,6.00


In [25]:
df.dtypes #data types

Season          str
Date            str
HomeTeam        str
AwayTeam        str
FTR             str
FTHG          int64
FTAG          int64
HS            int64
AS            int64
HST           int64
AST           int64
HC            int64
AC            int64
B365H       float64
B365D       float64
B365A       float64
dtype: object

In [26]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, format="mixed") #text to real dates and time format fix
df = df.sort_values("Date").reset_index(drop=True) #order match 
df.groupby("Season")["Date"].agg(["min", "max"]) #first vs last

,min,max
Season,,
1516,2015-08-08,2016-05-17
1617,2016-08-13,2017-05-21
1718,2017-08-11,2018-05-13
1819,2018-08-10,2019-05-12
1920,2019-08-09,2020-07-26
2021,2020-09-12,2021-05-23
2122,2021-08-13,2022-05-22
2223,2022-08-05,2023-05-28
2324,2023-08-11,2024-05-19


In [27]:
df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"]).sum() #duplicates

np.int64(0)

In [28]:
#target is 1 is home win and 0 is draw or away win
df["HomeWin"] = (df["FTR"] == "H").astype(int)

print(df["FTR"].value_counts(normalize=True))
print(df["HomeWin"].value_counts(normalize=True))

FTR
H    0.445000
A    0.321842
D    0.233158
Name: proportion, dtype: float64
HomeWin
0    0.555
1    0.445
Name: proportion, dtype: float64


No duplicated were found. Home teams won 44.5% of all the matches so the target is pretty balanced (55.5/44.5). If we always predicted "not home win", this would be 55.5% accurate.

In [29]:
df.describe().round(2) #simple quick summary of number columns

,Date,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A,HomeWin
count,3800,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00,3800.00
mean,2020-07-08 15:48:07.578947,1.55,1.28,13.92,11.50,4.78,4.02,5.72,4.72,2.96,4.28,4.76,0.44
min,2015-08-08 00:00:00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.06,2.80,1.12,0.00
25%,2017-12-25 06:00:00,1.00,0.00,10.00,8.00,3.00,2.00,3.00,3.00,1.65,3.48,2.20,0.00
50%,2020-08-19 00:00:00,1.00,1.00,13.00,11.00,5.00,4.00,5.00,4.00,2.25,3.75,3.30,0.00
75%,2023-01-21 00:00:00,2.00,2.00,17.00,15.00,6.00,5.00,8.00,6.00,3.25,4.50,5.50,1.00
max,2025-05-25 00:00:00,9.00,9.00,37.00,37.00,17.00,15.00,19.00,19.00,23.00,17.00,41.00,1.00
std,NaN,1.32,1.21,5.74,5.05,2.65,2.37,3.11,2.78,2.25,1.43,4.26,0.50


In [30]:
df[(df["HS"] == 0) | (df["AS"] == 0)] #checking 0 shots incase of data error

,Season,Date,HomeTeam,AwayTeam,FTR,FTHG,FTAG,HS,AS,HST,AST,HC,AC,B365H,B365D,B365A,HomeWin
1054,1718,2018-03-10,Huddersfield,Swansea,D,0,0,30,0,4,0,12,0,2.54,3.1,3.20,0
1422,1819,2019-03-02,Bournemouth,Man City,A,0,1,0,23,0,7,0,14,13.00,7.5,1.22,0


There were no impossible values were found such as negative goals and all odds above 1.0. Extreme values like 9 goals or zero shot performances and odds like 41.0 are actual real one sided games and performances. 

## 4. Preprocessing and feature engineering 

In [31]:
# One row per team per match. home teams view and away teams view
home = pd.DataFrame({
    "MatchID": df.index, #links back to og match
    "Date": df["Date"],
    "Team": df["HomeTeam"],
    "GoalsFor": df["FTHG"],
    "GoalsAgainst": df["FTAG"],
    "Shots": df["HS"],
    "ShotsOnTarget": df["HST"],
    "Corners": df["HC"],
    "Points": df["FTR"].map({"H": 3, "D": 1, "A": 0}),
})

away = pd.DataFrame({
    "MatchID": df.index,
    "Date": df["Date"],
    "Team": df["AwayTeam"],
    "GoalsFor": df["FTAG"],
    "GoalsAgainst": df["FTHG"],
    "Shots": df["AS"],
    "ShotsOnTarget": df["AST"],
    "Corners": df["AC"],
    "Points": df["FTR"].map({"H": 0, "D": 1, "A": 3}),
})

team_games = pd.concat([home, away]).sort_values(["Team", "Date"]).reset_index(drop=True) #put all together
team_games.head(10) #show only first few

,MatchID,Date,Team,GoalsFor,GoalsAgainst,Shots,ShotsOnTarget,Corners,Points
0,6,2015-08-09,Arsenal,0,2,22,6,5,0
1,17,2015-08-16,Arsenal,2,1,20,7,6,3
2,29,2015-08-24,Arsenal,0,0,19,5,7,1
3,35,2015-08-29,Arsenal,1,0,22,9,9,3
4,43,2015-09-12,Arsenal,2,0,29,12,13,3
5,55,2015-09-19,Arsenal,0,2,9,2,5,0
6,60,2015-09-26,Arsenal,5,2,26,12,5,3
7,77,2015-10-04,Arsenal,3,0,12,5,3,3
8,80,2015-10-17,Arsenal,3,0,17,6,9,3
9,90,2015-10-24,Arsenal,2,1,20,5,7,3


In [32]:
stat_cols = ["GoalsFor", "GoalsAgainst", "Shots", "ShotsOnTarget", "Corners", "Points"]


form = (
    team_games.groupby("Team")[stat_cols]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean()) 
)
#for each team average last 5 games not including current match preventing data leakage.
#Early games use however many past games exist

form.columns = ["Form_" + c for c in stat_cols] #new columns

team_games = pd.concat([team_games, form], axis=1)
team_games.head(7)

,MatchID,Date,Team,GoalsFor,GoalsAgainst,Shots,ShotsOnTarget,Corners,Points,Form_GoalsFor,Form_GoalsAgainst,Form_Shots,Form_ShotsOnTarget,Form_Corners,Form_Points
0,6,2015-08-09,Arsenal,0,2,22,6,5,0,NaN,NaN,NaN,NaN,NaN,NaN
1,17,2015-08-16,Arsenal,2,1,20,7,6,3,0.000000,2.00,22.000000,6.00,5.00,0.000000
2,29,2015-08-24,Arsenal,0,0,19,5,7,1,1.000000,1.50,21.000000,6.50,5.50,1.500000
3,35,2015-08-29,Arsenal,1,0,22,9,9,3,0.666667,1.00,20.333333,6.00,6.00,1.333333
4,43,2015-09-12,Arsenal,2,0,29,12,13,3,0.750000,0.75,20.750000,6.75,6.75,1.750000
5,55,2015-09-19,Arsenal,0,2,9,2,5,0,1.000000,0.60,22.400000,7.80,8.00,2.000000
6,60,2015-09-26,Arsenal,5,2,26,12,5,3,1.000000,0.60,19.800000,7.00,8.00,2.000000


In [33]:
form_cols = list(form.columns) #6 form columns
lookup = team_games[["MatchID", "Team"] + form_cols] #just whats needed to match form to matches

# Home version rename to HomeTeam so it lines up with df when merging
home_lookup = lookup.rename(columns={"Team": "HomeTeam"})
home_lookup = home_lookup.rename(columns={c: "Home" + c for c in form_cols}) # add Home to form columns so they dont clash with away


# Away version
away_lookup = lookup.rename(columns={"Team": "AwayTeam"})
away_lookup = away_lookup.rename(columns={c: "Away" + c for c in form_cols})

# Attach each teams form to its match
#re running adds columns
df["MatchID"] = df.index
df = df.merge(home_lookup, on=["MatchID", "HomeTeam"], how="left")
df = df.merge(away_lookup, on=["MatchID", "AwayTeam"], how="left")
df.shape 


(3800, 30)

## 5. Train/test split

In [34]:
#Clues for model to use
#all known before kickoff
odds_cols = ["B365H", "B365D", "B365A"]
form_feature_cols = ["Home" + c for c in form_cols] + ["Away" + c for c in form_cols]
feature_cols = odds_cols + form_feature_cols #3 odds and 12 form features so 15 total

# Time based split. 8 oldest seasons to train 2 newest seasons
train_seasons = SEASONS[:8]
test_seasons = SEASONS[8:]

train = df[df["Season"].isin(train_seasons)]
test = df[df["Season"].isin(test_seasons)]

# X is clues and Y is ans
X_train, y_train = train[feature_cols], train["HomeWin"]
X_test, y_test = test[feature_cols], test["HomeWin"]

print(X_train.shape, X_test.shape)

(3040, 15) (760, 15)


My data is split by time and not randomly. The 8 oldest seasons (2015/16 - 2022/23 with 3040 matches) are used for training and the two newest ones (2023/24 - 2024/25 with 760 matches) for testing. This matches how a model would be used in real life where it learns from past matches to predict ones in the future. This also doesnt allow it to learn from the future matches. The test set is kept untouched until the final evaluation.

In [35]:
print(X_train.isna().sum()) #counts empty cells in each column
print("Test missing:", X_test.isna().sum().sum())

B365H                      0
B365D                      0
B365A                      0
HomeForm_GoalsFor         17
HomeForm_GoalsAgainst     17
HomeForm_Shots            17
HomeForm_ShotsOnTarget    17
HomeForm_Corners          17
HomeForm_Points           17
AwayForm_GoalsFor         15
AwayForm_GoalsAgainst     15
AwayForm_Shots            15
AwayForm_ShotsOnTarget    15
AwayForm_Corners          15
AwayForm_Points           15
dtype: int64
Test missing: 12


Only a very few training matches have missing form. This is when each team first appears in the data. These are filled in the preprocessing pipeline.

In [ ]:
#prepare data before training
from sklearn.pipeline import Pipeline #combine
from sklearn.impute import SimpleImputer #fill missing values
from sklearn.preprocessing import StandardScaler #standerdise data

# preprocessing - fill missing form with median then scale
preprocess = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])



X_train_prep = preprocess.fit_transform(X_train) # Learn medians and scaling from the training data only
X_test_prep = preprocess.transform(X_test) # Apply what was learnt from training to test

print("Missing after:", pd.isna(X_train_prep).sum(), pd.isna(X_test_prep).sum()) #should be 0 0

Missing after: 0 0


Proprocessing pipeline - Missing form values are filled with the training data median, then all features are standerdised. The pipeline is only fitted to the training data and applied after to the test data. This ensures that no info from the test seasons leaks into the training.

## 6. Baselines 

In [40]:
from sklearn.dummy import DummyClassifier #baseline model guesses most common ans
from sklearn.model_selection import TimeSeriesSplit, cross_validate #split training into time ordered rounds + run and score

tscv = TimeSeriesSplit(n_splits=5)   # 5 rounds always in time order
scoring = ["accuracy", "f1", "roc_auc"]

#Baseline 1 which always predicts most common class (not home win)
dummy = Pipeline([("prep", preprocess), ("model", DummyClassifier(strategy="most_frequent"))])
dummy_scores = cross_validate(dummy, X_train, y_train, cv=tscv, scoring=scoring)

for metric in scoring:
    s = dummy_scores["test_" + metric]
    print(f"{metric}: {s.mean():.3f} ( +/- {s.std():.3f})")

accuracy: 0.546 ( +/- 0.037)
f1: 0.000 ( +/- 0.000)
roc_auc: 0.500 ( +/- 0.000)


In [41]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Baseline 2 which backs the bookies favourite (fixed rule no learning)
def bookies_predict(X):
    implied = 1 / X[["B365H", "B365D", "B365A"]]           # odds - rough probabilities
    prob_home = implied["B365H"] / implied.sum(axis=1)     # bookies home-win chance
    pred = (X["B365H"] < X[["B365D", "B365A"]].min(axis=1)).astype(int)  # 1 if home is favourite
    return pred, prob_home

results = {"accuracy": [], "f1": [], "roc_auc": []}
for _, check_idx in tscv.split(X_train):     # same 5 check chunks as baseline 1
    X_check, y_check = X_train.iloc[check_idx], y_train.iloc[check_idx]
    pred, prob = bookies_predict(X_check)
    results["accuracy"].append(accuracy_score(y_check, pred))
    results["f1"].append(f1_score(y_check, pred))
    results["roc_auc"].append(roc_auc_score(y_check, prob))

for metric, s in results.items():
    print(f"{metric}: {np.mean(s):.3f} (+/- {np.std(s):.3f})")

accuracy: 0.643 (+/- 0.026)
f1: 0.672 (+/- 0.033)
roc_auc: 0.735 (+/- 0.017)


In [43]:
#Show as table
baseline_table = pd.DataFrame({
    "Always not home win": {m: dummy_scores["test_" + m].mean() for m in scoring},
    "Bookies' favourite": {m: np.mean(results[m]) for m in scoring},
}).T.round(3)
baseline_table 

,accuracy,f1,roc_auc
Always not home win,0.546,0.000,0.500
Bookies' favourite,0.643,0.672,0.735


Baselines were scored with 5 fold time ordered cross validation on the training seasons only and the test seasons were not used. Baseline 1 always predicting “not home win” gives 54.6% accuracy but an F1 of 0 because it never predicts a home win. It also cant rank matches (ROC AUC = 0.5). Baseline 2, where backing the bookmakers favourite is much stronger (64.3% accuracy, ROC AUC 0.735, F1 = 0.672). My models will need to beat the first baseline, and beating the second one would give a very good result.